# t-stack-trajectory — ex1: stack per-step latents into a (B, T, D) trajectory

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `t-stack-trajectory`. Running the final beacon cell reports progress against the `Generative: torch.stack trajectory` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: torch.stack trajectory` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`t-stack-trajectory`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "t-stack-trajectory"
DD_SUBTOPIC = "Generative: torch.stack trajectory"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `torch.stack` trajectory — quick refresher

When you want to keep a TIME SERIES of intermediate tensors (per-step latents, per-epoch losses, per-iteration samples), the idiom is:

```python
history = []                          # python list
for step in range(T):
    z = compute_latent_at_step(step)  # (B, D)
    history.append(z)
trajectory = t.stack(history, dim=1)  # (B, T, D)
```

**Why `dim=1`, not `dim=0`.** `dim=0` produces `(T, B, D)` — time is the outermost axis. `dim=1` produces `(B, T, D)` — preserves batch as the outermost, time becomes the second axis. The latter is what most downstream tools expect (it matches `(batch, sequence, features)` from RNN/transformer conventions).

**Stack vs cat.** `t.stack` introduces a NEW axis at `dim`. `t.cat` concatenates along an EXISTING axis. If each `z` is `(B, D)` and you want `(B, T, D)`, you must `stack` — `cat(history, dim=1)` would give `(B, T*D)` instead.

**All tensors must match shape and dtype.** `t.stack` raises if they don't. Easy bug: appending a `(B, D)` and a `(D,)` from the same loop by accident.

### Exercise 1 — stack per-step latents into a (B, T, D) trajectory

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `torch.stack(latents_list, dim=1)` to convert a length-T Python list of `(B, D)` tensors into a single `(B, T, D)` trajectory tensor with TIME as the second axis.
> Keywords: stack, trajectory, time-axis, history
> ```

**KCs targeted:** `stack-new-axis-dim1`, `stack-list-of-equal-shapes`

Implement `ex1_stack_trajectory(per_step_latents)`. The collect-history pattern that ARENA's training loops use to log per-step intermediate latents:

1. `per_step_latents` is a Python list of length `T`. Each element is a `(B, D)` tensor (all the same shape).
2. Use `t.stack(per_step_latents, dim=1)` to stack along a NEW axis at position 1.
3. The result has shape `(B, T, D)` — batch first, then time, then feature.
4. **DO NOT** use `dim=0` (that would give `(T, B, D)`) and **DO NOT** use `t.cat` (that would concatenate, not stack).

Input: list of `T` tensors each shape `(B, D)`.
Output: `(B, T, D)` tensor.

The visualization picks the first batch element and plots each of its D latent dimensions as a separate line across T — you should see T-step trajectories per dimension.

In [ ]:
def ex1_stack_trajectory(per_step_latents: list[Tensor]) -> Tensor:
    """Stack a list of (B, D) tensors into a (B, T, D) trajectory."""
    raise NotImplementedError()


def _test_ex1():
    # Tiny exact case.
    step0 = t.tensor([[1.0, 2.0], [3.0, 4.0]])   # (B=2, D=2)
    step1 = t.tensor([[5.0, 6.0], [7.0, 8.0]])
    step2 = t.tensor([[9.0, 10.], [11., 12.]])
    out = ex1_stack_trajectory([step0, step1, step2])
    assert out.shape == (2, 3, 2), f'expected (B=2, T=3, D=2), got {tuple(out.shape)}'
    # Batch element 0's trajectory.
    expected_b0 = t.tensor([[1.0, 2.0], [5.0, 6.0], [9.0, 10.0]])
    assert t.allclose(out[0], expected_b0), f'batch 0 trajectory mismatch:\n{out[0]}'
    # Batch element 1's trajectory.
    expected_b1 = t.tensor([[3.0, 4.0], [7.0, 8.0], [11.0, 12.0]])
    assert t.allclose(out[1], expected_b1), f'batch 1 trajectory mismatch:\n{out[1]}'

    # Wrong-axis catch — make sure user used dim=1, not dim=0.
    # Output[B, T, D] = step_T[B, D]. If they did dim=0 they'd get shape (T, B, D)
    # and the (T, B, D)[0] slice would equal step0. Detect via shape mismatch.
    assert out[0].shape == (3, 2), (
        f'out[0].shape={tuple(out[0].shape)} — expected (T=3, D=2). '
        'If you got (B=2, D=2) you stacked along dim=0 instead of dim=1.'
    )

    # Realistic shape.
    rng = t.Generator().manual_seed(0)
    B, D, T_steps = 8, 16, 20
    history = [t.randn(B, D, generator=rng) for _ in range(T_steps)]
    traj = ex1_stack_trajectory(history)
    assert traj.shape == (B, T_steps, D), f'expected ({B},{T_steps},{D}), got {tuple(traj.shape)}'
    # Spot-check several (b, t, d) cells.
    for b in [0, 3, 7]:
        for tt in [0, 5, 19]:
            for d in [0, 8, 15]:
                assert traj[b, tt, d] == history[tt][b, d], f'cell ({b},{tt},{d}) mismatch'

    # Single-step degenerate — T=1 should give shape (B, 1, D).
    one_step = [t.randn(4, 5, generator=t.Generator().manual_seed(0))]
    one_traj = ex1_stack_trajectory(one_step)
    assert one_traj.shape == (4, 1, 5), f'T=1 must give (B, 1, D), got {tuple(one_traj.shape)}'

    # --- Visualization: per-dimension trajectory of batch element 0 ---
    fig, ax = plt.subplots(figsize=(7, 3))
    for d in range(D):
        ax.plot(range(T_steps), traj[0, :, d].numpy(), alpha=0.6)
    ax.set_xlabel('step (t)')
    ax.set_ylabel('latent value')
    ax.set_title(f'ex1 batch-element-0 trajectory across {T_steps} steps ({D} latent dims)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_stack_trajectory(per_step_latents: list[Tensor]) -> Tensor:
    return t.stack(per_step_latents, dim=1)
```

**Why `dim=1`, not `dim=0`.** Both are technically correct stacks — they just disagree on which axis becomes time. `dim=1` gives `(B, T, D)`, which matches the `(batch, sequence, features)` convention used by every recurrent / sequence module in PyTorch. `dim=0` gives `(T, B, D)`, which is what some older recurrence code expects but is otherwise idiosyncratic.

**Why `stack`, not `cat`.** `stack` introduces a NEW axis at `dim`. `cat` concatenates along an EXISTING axis. If you `cat` a list of `(B, D)` along `dim=1`, you get `(B, T*D)` — the time and feature axes get fused, which is almost never what you want.

**Shape rigidity.** Every tensor in the list must have the SAME shape (otherwise `stack` raises). A common bug: appending a `(B, D)` and a `(D,)` from the same loop (e.g. by indexing with `[0]` instead of `[:1]` for batch=1) — `stack` will fail with a fairly readable error.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()